In [10]:
import pandas as pd
import tensorflow as tf
import sklearn
import scikeras
import time
from scikeras.wrappers import KerasRegressor
from tensorflow.keras import backend as k
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn import metrics #imports

In [11]:
base = pd.read_csv('autos.csv', encoding='latin1') #coloquei o encoding pq tava dando erro, lendo o csv
base 

,dateCrawled,name,seller,offerType,price,abtest,vehicleType,yearOfRegistration,gearbox,powerPS,model,kilometer,monthOfRegistration,fuelType,brand,notRepairedDamage,dateCreated,nrOfPictures,postalCode,lastSeen
0,2016-03-24 11:52:17,Golf_3_1.6,privat,Angebot,480,test,NaN,1993,manuell,0,golf,150000,0,benzin,volkswagen,NaN,2016-03-24 00:00:00,0,70435,2016-04-07 03:16:57
1,2016-03-24 10:58:45,A5_Sportback_2.7_Tdi,privat,Angebot,18300,test,coupe,2011,manuell,190,NaN,125000,5,diesel,audi,ja,2016-03-24 00:00:00,0,66954,2016-04-07 01:46:50
2,2016-03-14 12:52:21,"Jeep_Grand_Cherokee_""Overland""",privat,Angebot,9800,test,suv,2004,automatik,163,grand,125000,8,diesel,jeep,NaN,2016-03-14 00:00:00,0,90480,2016-04-05 12:47:46
3,2016-03-17 16:54:04,GOLF_4_1_4__3TÜRER,privat,Angebot,1500,test,kleinwagen,2001,manuell,75,golf,150000,6,benzin,volkswagen,nein,2016-03-17 00:00:00,0,91074,2016-03-17 17:40:17
4,2016-03-31 17:25:20,Skoda_Fabia_1.4_TDI_PD_Classic,privat,Angebot,3600,test,kleinwagen,2008,manuell,69,fabia,90000,7,diesel,skoda,nein,2016-03-31 00:00:00,0,60437,2016-04-06 10:17:21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371523,2016-03-14 17:48:27,Suche_t4___vito_ab_6_sitze,privat,Angebot,2200,test,NaN,2005,NaN,0,NaN,20000,1,NaN,sonstige_autos,NaN,2016-03-14 00:00:00,0,39576,2016-04-06 00:46:52
371524,2016-03-05 19:56:21,Smart_smart_leistungssteigerung_100ps,privat,Angebot,1199,test,cabrio,2000,automatik,101,fortwo,125000,3,benzin,smart,nein,2016-03-05 00:00:00,0,26135,2016-03-11 18:17:12
371525,2016-03-19 18:57:12,Volkswagen_Multivan_T4_TDI_7DC_UY2,privat,Angebot,9200,test,bus,1996,manuell,102,transporter,150000,3,diesel,volkswagen,nein,2016-03-19 00:00:00,0,87439,2016-04-07 07:15:26
371526,2016-03-20 19:41:08,VW_Golf_Kombi_1_9l_TDI,privat,Angebot,3400,test,kombi,2002,manuell,100,golf,150000,6,diesel,volkswagen,NaN,2016-03-20 00:00:00,0,40764,2016-03-24 12:45:21


In [14]:
inicio = time.time() #inciiando tempo

In [16]:
inicio

1779640830.6029036

In [12]:
base = base.drop('dateCrawled', axis=1)
base = base.drop('dateCreated', axis=1)
base = base.drop('nrOfPictures', axis=1)
base = base.drop('postalCode', axis=1)
base = base.drop('lastSeen', axis=1)
base = base.drop('name', axis=1)
base = base.drop('seller', axis=1)
base = base.drop('offerType', axis=1) #removendo colunas

In [17]:
base = base[base.price > 10]
base = base.loc[base.price < 350000] #pegando somente alguns precos

In [22]:
valores = {'vehicleType': 'limousine',
    'gearbox': 'manuell',
    'model': 'golf',
    'fuelType': 'benzin',
    'notRepairedDamage': 'nein'
} #valores
base = base.fillna(value = valores) #preenchendo os NaN

In [23]:
X = base.iloc[:, 1:12].values
y = base.iloc[:, 0].values #separando os valores

In [24]:
X #vendo x

array([['test', 'limousine', 1993, ..., 'benzin', 'volkswagen', 'nein'],
       ['test', 'coupe', 2011, ..., 'diesel', 'audi', 'ja'],
       ['test', 'suv', 2004, ..., 'diesel', 'jeep', 'nein'],
       ...,
       ['test', 'bus', 1996, ..., 'diesel', 'volkswagen', 'nein'],
       ['test', 'kombi', 2002, ..., 'diesel', 'volkswagen', 'nein'],
       ['control', 'limousine', 2013, ..., 'benzin', 'bmw', 'nein']],
      shape=(359291, 11), dtype=object)

In [25]:
y #vendo y

array([  480, 18300,  9800, ...,  9200,  3400, 28990], shape=(359291,))

In [26]:
onehotencoder = ColumnTransformer(transformers=[('OneHot', OneHotEncoder(), [0, 1, 3, 5, 8, 9, 10])], remainder='passthrough') #passando dado categorico pra numerico
X = onehotencoder.fit_transform(X).toarray() #transforma pra array

In [27]:
X.shape #shape do X

(359291, 316)

In [28]:
def criar_rede():
    k.clear_session()
    regressor = Sequential([
        tf.keras.layers.InputLayer(shape=(316,)),
        tf.keras.layers.Dense(units=158, activation='relu'),
        tf.keras.layers.Dense(units=158, activation='relu'),
        tf.keras.layers.Dense(units=1, activation='linear')
    ]) #cria a rede 
    regressor.compile(loss='mean_absolute_error', optimizer='adam', metrics=['mean_absolute_error']) #compila
    return regressor

In [30]:
regressor = KerasRegressor(model = criar_rede, epochs = 100, batch_size = 300) #regressor

In [31]:
resultados = cross_val_score(estimator = regressor, X = X, y = y, cv = 5, scoring='neg_mean_absolute_error') #dividindo

W0000 00:00:1779641292.906323   72771 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Epoch 1/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 3870.3101 - mean_absolute_error: 3870.3101
Epoch 2/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 3345.2532 - mean_absolute_error: 3345.2532
Epoch 3/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 3072.1558 - mean_absolute_error: 3072.1558
Epoch 4/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 2904.2178 - mean_absolute_error: 2904.2178
Epoch 5/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2852.9800 - mean_absolute_error: 2852.9797
Epoch 6/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 2794.5208 - mean_absolute_error: 2794.5208
Epoch 7/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 2745.7815 - mean_absolute_error: 2745.7815
Epoch 8/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 2720.6697 - mean_absolute_error: 2720.6697
Epoch 9/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 2698.6421 - mean_absolute_error: 2698.6421
Epoch 10/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 26

In [32]:
fim = time.time() #pegando o fim

In [33]:
fim

1779644136.3918087

In [35]:
(fim - inicio) / 60 / 60 #acabou em 0.91 hora

0.9182746958732605

In [36]:
abs(resultados) #pegando o erro absoluto em cada execucao

array([2435.92993164, 2256.41259766, 2222.79541016, 2617.9519043 ,
       2436.8684082 ])

In [37]:
abs(resultados.mean()) #media absoluto

np.float64(2393.991650390625)

In [39]:
resultados.std() #desvio padrao

np.float64(142.82267295226038)